# Strands Decider 2B on Colab

Select **Runtime → Change runtime type → GPU** before running. The first model call downloads the public model weights. The one-shot smoke check and protected HTTP tunnel below do not call Bedrock. For the tunnel, add `NGROK_AUTHTOKEN`, `DECIDER_TUNNEL_USERNAME`, and `DECIDER_TUNNEL_PASSWORD` in Colab Secrets first. Use an alphanumeric username/password and grant this notebook access to those secrets.

In [1]:
!pip install -q "strands-decider[cuda] @ git+https://github.com/strands-labs/strands-decider.git" ngrok

In [2]:
import torch
assert torch.cuda.is_available(), 'Choose Runtime → Change runtime type → GPU, then reconnect.'
print(torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory // 2**30, 'GiB VRAM')

NVIDIA GeForce GTX 1650 3 GiB VRAM


In [3]:
!strands-decider ask StrandsAgents/strands-decider-2B-qwen3.5-v1-2610 \
  --device cuda \
  --state "A test failed twice; the developer has not opened the traceback." \
  --noul "Would a thoughtful teammate interject right now?" \
  --json

Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 29 files: 100%|█████████████████████| 29/29 [00:00<00:00, 1361.39it/s]
Download complete:                                         |  0.00B            
Download complete:                                         |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            
Loading weights: 100%|█████████████████████| 320/320 [00:00<00:00, 1259.45it/s]
[W1016 08:45:25.194441445 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 25165824 bytes (free: 4128768, total: 3895132160).
[W1016 08:45:25.197377700 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 25165824 bytes (free: 8323072, total: 3895132160).
╭───────────────────── Traceback (most recent call last) ──────────────────────╮
│ /home/shivansh/Desktop/The-Agent-That-Knows-When-to-Shut-Up/.venv/lib/python │

## Start the model server

Run this cell once and leave the Colab runtime connected. It starts the HTTP API expected by this repository's `HttpDecider` client.

In [5]:
import socket
import subprocess
import time

MODEL_ID = 'StrandsAgents/strands-decider-2B-qwen3.5-v1-2610'
server_log = open('/tmp/strands-decider.log', 'w')
decider_process = subprocess.Popen(
    ['strands-decider', 'serve', MODEL_ID, '--device', 'cuda', '--port', '8000'],
    stdout=server_log, stderr=subprocess.STDOUT,
)
deadline = time.monotonic() + 300
while time.monotonic() < deadline:
    if decider_process.poll() is not None:
        raise RuntimeError(open('/tmp/strands-decider.log').read()[-4000:])
    try:
        with socket.create_connection(('127.0.0.1', 8000), timeout=1):
            print('Strands Decider server is listening on port 8000')
            break
    except OSError:
        time.sleep(2)
else:
    decider_process.terminate()
    raise TimeoutError('Decider server did not start; inspect /tmp/strands-decider.log')

RuntimeError: classes[0m                                                     [31m│[0m
[31m│[0m   [2m 936 [0m[2m│   │   [0m[2m# is in dynamo's MOD_SKIPLIST, revisit later for c++[0m          [31m│[0m
[31m│[0m                                                                              [31m│[0m
[31m│[0m [2m/home/shivansh/Desktop/The-Agent-That-Knows-When-to-Shut-Up/.venv/lib/python[0m [31m│[0m
[31m│[0m [2m3.12/site-packages/torch/nn/modules/[0m[1mmodule.py[0m:966 in _apply                  [31m│[0m
[31m│[0m                                                                              [31m│[0m
[31m│[0m   [2m 963 [0m[2m│   │   │   [0m[2m# track autograd history of `param_applied`, so we have [0m  [31m│[0m
[31m│[0m        [2mto use[0m                                                                [31m│[0m
[31m│[0m   [2m 964 [0m[2m│   │   │   [0m[2m# `with torch.no_grad():`[0m                                 [31m│[0m
[31m│[0m   [

## Open a password-protected HTTPS tunnel

The endpoint URL printed below is public, but the Decider API requires the Basic Auth credentials stored in Colab Secrets. Copy the URL and credentials into the local VS Code bridge configuration. Keep the Colab runtime running while testing.

In [ ]:
import os
import re
import ngrok
from google.colab import userdata

os.environ['NGROK_AUTHTOKEN'] = userdata.get('NGROK_AUTHTOKEN')
tunnel_user = userdata.get('DECIDER_TUNNEL_USERNAME')
tunnel_password = userdata.get('DECIDER_TUNNEL_PASSWORD')
safe_credential = re.compile(r'^[A-Za-z0-9_-]{16,}$')
if not safe_credential.fullmatch(tunnel_user) or not safe_credential.fullmatch(tunnel_password):
    raise ValueError('Use at least 16 letters, digits, underscores, or hyphens for both tunnel credentials')
traffic_policy = f'''on_http_request:
  - actions:
      - type: basic-auth
        config:
          credentials:
            - {tunnel_user}:{tunnel_password}
'''
decider_tunnel = ngrok.forward(
    'localhost:8000', authtoken_from_env=True, traffic_policy=traffic_policy
)
DECIDER_URL = decider_tunnel.url()
print('DECIDER_URL=' + DECIDER_URL)
print('Copy DECIDER_URL, DECIDER_TUNNEL_USERNAME, and DECIDER_TUNNEL_PASSWORD into your local terminal.')